# Smoke Test - Reliable Assistive VQA

**RUNTIME: CPU is fine. No GPU, no Drive, no dataset required.**

Exercises every `src/` code path with synthetic data + a monkeypatched backbone,
so you catch Python/logic errors **before** spending compute units on the real pipeline.

Run all cells top-to-bottom. Each prints `[PASS]`/`[FAIL]`. The last cell asserts all passed.
If a cell fails, copy the traceback to Claude to fix the matching `src/` module.

## 0. Setup - clone repo + minimal deps (CPU)

In [ ]:
# RUNTIME: CPU is fine. Clones from GitHub so you always test the latest push.
import os, sys, subprocess
REPO_URL='https://github.com/meteorboyF/VQA-paper.git'
REPO_ROOT='/content/VQA-paper'
if os.path.isdir('/content'):
    if not os.path.exists(REPO_ROOT):
        subprocess.run(['git','clone',REPO_URL,REPO_ROOT],check=True)
    else:
        subprocess.run(['git','-C',REPO_ROOT,'pull'],check=True)
    os.chdir(REPO_ROOT)
else:
    REPO_ROOT=os.getcwd()
if REPO_ROOT not in sys.path: sys.path.insert(0,REPO_ROOT)
# Smoke test needs only the light stack (all preinstalled in Colab):
subprocess.run([sys.executable,'-m','pip','install','-q','pyarrow'],check=False)
print('setup ok; cwd=',os.getcwd())

## 1. Test harness + synthetic data

In [ ]:
import os, json, tempfile, shutil
import numpy as np, torch
from src.data_assembly import QUALITY_FLAWS
DEFECT_NAMES=QUALITY_FLAWS+['unrecognizable']; N_DEF=len(DEFECT_NAMES)
DEVICE='cpu'; FAILS=[]
def check(name, fn):
    try:
        fn(); print(f'  [PASS] {name}')
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f'  [FAIL] {name}: {type(e).__name__}: {e}'); FAILS.append(name)
WORK=tempfile.mkdtemp(prefix='vqa_smoke_'); print('scratch:',WORK)
rng=np.random.RandomState(7); DIM=512
def synth(n,seed):
    r=np.random.RandomState(seed); X=r.randn(n,DIM).astype(np.float32)
    y=(X[:,0]+r.randn(n)*0.5>0).astype(np.float32); Y=np.zeros((n,N_DEF),np.float32)
    for d in range(N_DEF): Y[:,d]=(X[:,d+1]+r.randn(n)*0.6>0.8).astype(np.float32)
    return X,y,Y
Xtr,ytr,Ytr=synth(400,1); Xca,yca,Yca=synth(120,2); Xre,yre,Yre=synth(280,3)
print('synthetic data ready')

## 2. Imports - every src module + experiment modules + expstate

In [ ]:
def _imp():
    from src import (env,config,data_assembly,heads,train_eval,stats,
        calibration,selective,vqa_confidence,actionable,grounding,figures,resultlog,features,
        expstate,staging,progress)
check('import all 17 src modules', _imp)
def _imp_exp():
    import pkgutil, importlib
    import src.experiments as _pkg
    mods = [m.name for m in pkgutil.iter_modules(_pkg.__path__)]
    for m in mods:
        importlib.import_module(f'src.experiments.{m}')
    print(f'    ({len(mods)} experiment modules)')
check('import ALL experiment modules', _imp_exp)
from src import (env,config,data_assembly,heads,train_eval,stats,
    calibration,selective,vqa_confidence,actionable,grounding,figures,resultlog,features)

# expstate: DONE marker round-trip honoring required artifacts
def _expstate():
    from src import expstate
    d=os.path.join(WORK,'resE'); os.makedirs(d,exist_ok=True)
    art=os.path.join(d,'thing.json'); open(art,'w').write('{}')
    assert not expstate.is_done('EX',d,required=[art])
    expstate.mark_done('EX',d,artifacts=[art])
    assert expstate.is_done('EX',d,required=[art])
    os.remove(art)
    assert not expstate.is_done('EX',d,required=[art])  # missing artifact => rerun
check('expstate DONE-marker skip logic', _expstate)


## 3. env - seeding, cal/rep split, frozen-knob guard

In [ ]:
check('seed_everything', lambda: env.seed_everything(0))
def _split():
    vi=np.arange(200); strat=np.random.RandomState(0).randint(0,2,200)
    cal,rep=env.make_cal_rep_split(vi,0.3,stratify_labels=strat)
    assert len(cal)+len(rep)==200
    assert len(set(cal.tolist())&set(rep.tolist()))==0,'cal/rep overlap!'
    p=os.path.join(WORK,'split.json'); env.save_split_ids(cal,rep,p)
    c2,r2=env.load_split_ids(p); assert np.array_equal(cal,c2)
check('make/save/load cal-rep split (disjoint)', _split)
def _leak():
    env.assert_no_rep_leakage('cal')
    for bad in ('rep','test','report'):
        try: env.assert_no_rep_leakage(bad); raise AssertionError(bad+' did not raise')
        except ValueError: pass
check('frozen-knob assertion raises on rep/test', _leak)

## 4. data_assembly - build_master + schema robustness

In [ ]:
from src import data_assembly
def _mk_annot():
    vqa,qual=[],[]; r=np.random.RandomState(1)
    for i in range(60):
        img=f'VizWiz_val_{i:08d}.jpg'
        vqa.append({'image':img,'question':f'what is this {i}?',
            'answerable':int(r.rand()>0.3),'answers':[{'answer':'cat'}]*7+[{'answer':'dog'}]*3})
        qual.append({'image':img,'flaws':{f:int(r.rand()>0.7) for f in QUALITY_FLAWS},
            'unrecognizable':int(r.rand()>0.85)})
    vp=os.path.join(WORK,'vqa.json'); json.dump(vqa,open(vp,'w'))
    qp=os.path.join(WORK,'qual.json'); json.dump(qual,open(qp,'w')); return vp,qp
VP,QP=_mk_annot()
def _assemble():
    mp=os.path.join(WORK,'master.parquet')
    m=data_assembly.build_master({'val':VP},{'val':QP},mp); assert len(m)==60
    for c in [f'q_{f}' for f in DEFECT_NAMES]+['answerable','question','answers']:
        assert c in m.columns, 'missing '+c
    s=data_assembly.label_stats(m,os.path.join(WORK,'label_stats.json'))
    assert 'cooccurrence' in s and 'positive_rates' in s
check('build_master + label_stats (real schema shape)', _assemble)
def _alt():
    qual=[{'image':f'v{i:08d}.jpg','quality_flaws':['blur'] if i%2 else [],
           'not_recognizable':i%5==0} for i in range(20)]
    qp=os.path.join(WORK,'qual_alt.json'); json.dump(qual,open(qp,'w'))
    df=data_assembly.load_quality(qp,'val')
    assert df['q_blur'].sum()==10 and df['q_unrecognizable'].sum()==4
check('load_quality handles list-flaws + alt key names', _alt)

## 5. heads - Linear / MLP / Joint

In [ ]:
from src import heads
def _heads():
    x=torch.randn(8,512)
    assert heads.LinearHead(512,1)(x).shape==(8,1)
    assert heads.MLPHead(512,N_DEF,multilabel=True)(x).shape==(8,N_DEF)
    t,d=heads.JointHead(512,n_defect=N_DEF)(x); assert t.shape==(8,1) and d.shape==(8,N_DEF)
    for ht,do in [('linear',1),('mlp',N_DEF),('joint',N_DEF)]: heads.build_head(ht,512,do)
check('LinearHead/MLPHead/JointHead/build_head', _heads)

## 6. train_eval - training, threshold selection, metrics

In [ ]:
from src import train_eval
def _losses():
    pw=train_eval.compute_pos_weight(Ytr)
    for lv in ('bce','pos_weight','focal'):
        crit=train_eval.build_criterion(lv,pw,DEVICE)
        assert crit(torch.randn(8,N_DEF),torch.randint(0,2,(8,N_DEF)).float()).item()>=0
check('build_criterion bce/pos_weight/focal', _losses)
def _tb():
    env.seed_everything(0); m=heads.MLPHead(DIM,1)
    m,_=train_eval.train_head(m,Xtr,ytr,Xca,yca,seed=0,device=DEVICE,max_epochs=4,patience=3)
    lr=m(torch.tensor(Xre)).squeeze(-1).detach().numpy()
    lc=m(torch.tensor(Xca)).squeeze(-1).detach().numpy()
    t=train_eval.find_threshold(yca,lc,split_name='cal')
    met=train_eval.evaluate_binary(yre,lr,t); assert np.array(met['confusion_matrix']).shape==(2,2)
check('train_head + find_threshold(cal) + evaluate_binary', _tb)
def _guard():
    try: train_eval.find_threshold(yre,np.zeros(len(yre)),split_name='rep'); raise AssertionError('no guard')
    except ValueError: pass
check('find_threshold refuses split=rep', _guard)
def _ml():
    env.seed_everything(0); m=heads.MLPHead(DIM,N_DEF,multilabel=True)
    m,_=train_eval.train_head(m,Xtr,Ytr,Xca,Yca,seed=0,device=DEVICE,max_epochs=4,patience=3)
    lr=m(torch.tensor(Xre)).detach().numpy(); th=np.full(N_DEF,0.5)
    met=train_eval.evaluate_multilabel(Yre,lr,th,DEFECT_NAMES)
    assert 'mAP' in met and len(met['confusion_matrices_one_vs_rest'])==N_DEF
    for d in DEFECT_NAMES: assert np.array(met['confusion_matrices_one_vs_rest'][d]).shape==(2,2)
check('evaluate_multilabel (per-defect 2x2, mAP, macro/micro-F1)', _ml)
def _ms():
    res=train_eval.run_multi_seed(lambda:heads.MLPHead(DIM,1),Xtr,ytr,Xca,yca,Xre,yre,
        label_names=['answerable'],
        threshold_fn=lambda yc,lg:train_eval.find_threshold(yc,lg,split_name='cal'),
        eval_fn=lambda yr,lg,t:train_eval.evaluate_binary(yr,lg,t),
        seeds=[0,1],device=DEVICE,max_epochs=3,patience=2)
    assert 'AUROC' in res and 'mean' in res['AUROC'] and len(res['_logits_rep'])==2
check('run_multi_seed aggregates over seeds', _ms)

## 7. stats - bootstrap, paired delta, BH-FDR, DeLong

In [ ]:
from src import stats; from sklearn.metrics import roc_auc_score
def _st():
    y=rng.randint(0,2,300); a=rng.rand(300); b=rng.rand(300)
    m,lo,hi=stats.bootstrap_ci(lambda yy,ss:roc_auc_score(yy,ss),y,a,n_boot=200); assert lo<=m<=hi
    d,l,h,p=stats.paired_bootstrap_delta(lambda yy,ss:roc_auc_score(yy,ss),y,a,b,n_boot=200); assert 0<=p<=1
    rej=stats.benjamini_hochberg([0.001,0.04,0.2,0.5,0.9]); assert rej.dtype==bool and len(rej)==5
    ms=stats.multi_seed(lambda seed:{'acc':0.8+seed*0.01},seeds=[0,1,2]); assert 'ci95' in ms['acc']
check('bootstrap_ci/paired_delta/BH-FDR/multi_seed', _st)
def _dl():
    y=rng.randint(0,2,300); a=y*0.5+rng.rand(300)*0.5; b=rng.rand(300)
    aa,ab,dd,z,p=stats.delong_auroc(y,a,b); assert 0<=p<=1 and aa>ab
check('delong_auroc valid z/p', _dl)

## 8. calibration - temp scaling, ECE, defect-aware

In [ ]:
from src import calibration
def _cal():
    confs=rng.rand(300); correct=(rng.rand(300)<confs).astype(float)
    logits=np.log(confs.clip(1e-6,1-1e-6)/(1-confs.clip(1e-6,1-1e-6)))
    T=calibration.temperature_scale(logits,correct.astype(int),split_name='cal'); assert T>0
    pt=calibration.apply_temperature(logits,T); assert ((0<=pt)&(pt<=1)).all()
    assert 0<=calibration.ece(confs,correct)<=1
    assert 'bin_acc' in calibration.ece_diagram_data(confs,correct)
    assert 0<=calibration.brier_score(confs,correct)<=1
check('temperature_scale(cal)/apply/ece/brier', _cal)
def _cg():
    try: calibration.temperature_scale(np.zeros(10),np.zeros(10,int),split_name='rep'); raise AssertionError('no guard')
    except ValueError: pass
check('temperature_scale refuses split=rep', _cg)
def _dac():
    confs=rng.rand(300); correct=(rng.rand(300)<confs).astype(float); did=rng.randint(-1,N_DEF,300)
    sc=calibration.defect_aware_calibration(confs,correct,did,N_DEF,split_name='cal'); assert 'global' in sc
    out=calibration.apply_defect_aware_calibration(confs,did,sc); assert out.shape==confs.shape
check('defect_aware_calibration + apply', _dac)

## 9. selective - risk-coverage, AURC, gates (NumPy 2.x safe)

In [ ]:
from src import selective
def _sel():
    confs=rng.rand(300); correct=(rng.rand(300)<confs).astype(float)
    cov,risk=selective.risk_coverage_curve(confs,correct); assert len(cov)==len(risk)==300
    assert 0<=selective.aurc(confs,correct)<=1   # exercises np.trapz/trapezoid shim
    t=selective.find_global_threshold(confs,correct,0.8,split_name='cal')
    assert selective.global_gate(confs,t).dtype==bool
    did=rng.randint(-1,N_DEF,300)
    dth=selective.find_defect_thresholds(confs,correct,did,N_DEF,split_name='cal')
    assert selective.defect_conditioned_gate(confs,did,dth,t).dtype==bool
    assert 'aurc_global_raw' in selective.compare_policies(confs,confs,confs,correct)
    assert 'aurc' in selective.risk_coverage_for_figure(confs,correct,'x')
check('risk_coverage/aurc/gates/find_*_threshold(cal)', _sel)
def _sg():
    for fn in (lambda:selective.find_global_threshold(rng.rand(10),rng.rand(10),0.8,split_name='rep'),
               lambda:selective.find_defect_thresholds(rng.rand(10),rng.rand(10),rng.randint(-1,N_DEF,10),N_DEF,split_name='rep')):
        try: fn(); raise AssertionError('no guard')
        except ValueError: pass
check('selective threshold fns refuse split=rep', _sg)

## 10. vqa_confidence (metric) + actionable (ARR/FRR)

In [ ]:
from src import vqa_confidence, actionable
def _acc():
    assert vqa_confidence.vqa_accuracy('cat',['cat']*3)==1.0
    assert abs(vqa_confidence.vqa_accuracy('cat',['cat']*2+['dog']*8)-2/3)<1e-9
    assert vqa_confidence.vqa_accuracy('zzz',['cat']*10)==0.0
check('vqa_accuracy = min(#matches/3,1)', _acc)
def _arr():
    n=200; probs=rng.rand(n,N_DEF); gt=(rng.rand(n,N_DEF)>0.7).astype(int); ans=(rng.rand(n)>0.4).astype(int)
    res=actionable.actionable_recovery_rate(probs,gt,ans,DEFECT_NAMES)
    assert all(k in res for k in ('ARR','FRR','ARR_ci95','FRR_ci95','per_defect'))
    assert len(res['defect_to_action'])>=6
check('actionable_recovery_rate ARR+FRR+CI+per_defect', _arr)

## 11. grounding - groundability features + entity extraction (no model load)

In [ ]:
from src import grounding
def _gf():
    f=grounding.groundability_features({'grounded':True,'conf':0.9,'boxes':[[100,100,400,400]]},1000,1000)
    assert f['grounded']==1 and f['touches_border']==0 and 0<=f['centeredness']<=1
    fe=grounding.groundability_features({'grounded':True,'conf':0.5,'boxes':[[0,0,200,999]]},1000,1000)
    assert fe['touches_border']==1
    fn=grounding.groundability_features({'grounded':False,'conf':0.0,'boxes':[]},1000,1000)
    assert fn['grounded']==0 and fn['n_boxes']==0
check('groundability_features (centered/border/negative-block)', _gf)
def _ent():
    e=grounding.extract_entity('what color is the shirt?'); assert isinstance(e,str) and len(e)>0
    assert isinstance(grounding.extract_entity('how many cans are on the table?'),str)
check('extract_entity (spaCy or deterministic fallback)', _ent)

## 12. features - shard/cache/resume (monkeypatched backbone, no downloads)

In [ ]:
from src import features
def _feat():
    from PIL import Image
    import torchvision.transforms as T
    img_dir=os.path.join(WORK,'imgs'); os.makedirs(img_dir,exist_ok=True); paths=[]
    for i in range(20):
        p=os.path.join(img_dir,f'{i}.jpg'); Image.new('RGB',(32,32),(i*10%255,0,0)).save(p); paths.append(p)
    pre=T.Compose([T.Resize((16,16)),T.ToTensor()])
    class Tiny(torch.nn.Module):
        def forward(self,x): return x.flatten(1)[:,:8]
    orig=features.load_backbone
    features.load_backbone=lambda name,device:(Tiny().eval(),pre,8)
    try:
        out=os.path.join(WORK,'emb_tiny.npy')
        e=features.extract('tiny',paths,out,device='cpu',bs=4,num_workers=0,force=True)
        assert e.shape==(20,8) and e.dtype==np.float16
        e2=features.extract('tiny',paths,out,device='cpu',bs=4,num_workers=0,force=False)
        assert e2.shape==(20,8)
        features.build_feature_index(paths,[os.path.basename(p) for p in paths],['val']*20,os.path.join(WORK,'fi.parquet'))
    finally:
        features.load_backbone=orig
check('features.extract shard+cache+idempotent + feature_index', _feat)

## 13. resultlog - versioned JSON + manifest + RESULTS.md

In [ ]:
from src import resultlog
def _rl():
    rd=os.path.join(WORK,'results','E0_audit')
    p=resultlog.log_run('SMOKE',{'acc':0.9},{'seed':0},rd,repo_root=WORK)
    assert os.path.exists(p)
    assert os.path.exists(os.path.join(WORK,'results','manifest.jsonl'))
    assert os.path.exists(os.path.join(WORK,'results','RESULTS.md'))
check('log_run writes json+manifest+RESULTS.md', _rl)

## 14. figures - F1-F10 render + save PDF/PNG

In [ ]:
from src import figures
def _figs():
    fd=os.path.join(WORK,'figs'); figures.set_fig_dir(fd)
    figures.f1_pipeline_schematic()
    figures.f2_cooccurrence(os.path.join(WORK,'label_stats.json'))
    e4={bb:{'macro_F1':{'mean':.5,'std':.02},'micro_F1':{'mean':.5,'std':.02},'mAP':{'mean':.4,'std':.03},
            'per_defect_auroc':{d:{'mean':.7,'std':.05} for d in DEFECT_NAMES},
            'per_defect_auprc':{d:{'mean':.4,'std':.05} for d in DEFECT_NAMES}} for bb in ('clip','mobilenet')}
    e3={bb:{'AUROC':{'mean':.8,'std':.02},'AUPRC':{'mean':.7,'std':.03},'F1':{'mean':.6,'std':.02},'balanced_acc':{'mean':.7,'std':.02}} for bb in ('clip','mobilenet')}
    figures.f3_per_defect_auroc(e4)
    figures.f7_backbone_comparison({bb:{**e3[bb],**e4[bb]} for bb in e3})
    confs=rng.rand(200); correct=(rng.rand(200)<confs).astype(float)
    dd=calibration.ece_diagram_data(confs,correct)
    cj=os.path.join(WORK,'calib.json'); json.dump({'raw':{**dd,'ece':0.1},'temp':{**dd,'ece':0.05}},open(cj,'w'))
    figures.f4_reliability_diagram(cj)
    rc={k:selective.risk_coverage_for_figure(rng.rand(200),correct,k) for k in ('random','global_raw','global_temp','defect_aware')}
    rc['delta_p']=0.03; rcj=os.path.join(WORK,'rc.json'); json.dump(rc,open(rcj,'w')); figures.f5_risk_coverage(rcj)
    arr={'ARR':0.6,'FRR':0.2,'ARR_ci95':[0.5,0.7],'FRR_ci95':[0.1,0.3],'per_defect':{d:{'ARR':0.5,'n':10} for d in DEFECT_NAMES}}
    aj=os.path.join(WORK,'arr.json'); json.dump(arr,open(aj,'w')); figures.f6_arr_frr(aj)
    figures.f8_roc_panels({bb:{**e3[bb],'fpr':list(np.linspace(0,1,10)),'tpr':list(np.linspace(0,1,10)**0.5)} for bb in e3},
                          {d:{'fpr':list(np.linspace(0,1,10)),'tpr':list(np.linspace(0,1,10)**0.5),'AUROC':0.7} for d in DEFECT_NAMES})
    from PIL import Image
    qi=os.path.join(WORK,'qi.jpg'); Image.new('RGB',(64,64),(0,128,0)).save(qi)
    figures.f9_qualitative_grid([{'image_path':qi,'defect':'blur','split_type':'TP','label':'answerable'}]*4)
    td={'subsample_n':4000,'grounder':'locate_anything','delta_AUROC':0.03,'delta_AUPRC':0.02,
        'delta_AUROC_ci_lo':0.01,'delta_AUROC_ci_hi':0.05,'delta_AUPRC_ci_lo':0.0,'delta_AUPRC_ci_hi':0.04,
        'delta_AUROC_p':0.02,'delong_z':2.1,'delong_p':0.03}
    tj=os.path.join(WORK,'td.json'); json.dump(td,open(tj,'w')); figures.f10_groundability(tj)
    pdfs=[f for f in os.listdir(fd) if f.endswith('.pdf')]; assert len(pdfs)>=9, f'only {len(pdfs)} figs'
check('all figures F1-F10 render + save PDF/PNG', _figs)

## 15. Summary

In [ ]:
print('='*60)
if FAILS:
    print(f'SMOKE TEST: {len(FAILS)} FAILURE(S):')
    for f in FAILS: print('   -',f)
    print('Copy the tracebacks above to Claude to fix the src/ modules.')
else:
    print('SMOKE TEST: ALL PASSED. Engine is sound - safe to run the real pipeline.')
print('='*60)
shutil.rmtree(WORK, ignore_errors=True)
assert not FAILS, f'{len(FAILS)} smoke-test failures'